# Delta Lake Basics — 07: Schema Enforcement and Evolution

## What you will learn
Delta's schema management is one of its key differentiators.
By default it strictly enforces the schema — preventing silent data corruption.

In this notebook:
1. Schema enforcement — why writes are rejected
2. `mergeSchema` — opt-in schema evolution
3. `ALTER TABLE ADD COLUMNS` — the production way to add columns
4. Column renaming and type changes
5. Column mapping mode — rename without data rewrite
6. Schema evolution vs schema enforcement trade-offs


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:06:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Delta Lake ready


26/04/10 20:06:43 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:06:45 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


In [2]:
TABLE = f"{DATA_DIR}/07_schema"
shutil.rmtree(TABLE, ignore_errors=True)
df.write.format("delta").mode("overwrite").save(TABLE)
print("Table created with schema:")
spark.read.format("delta").load(TABLE).printSchema()

26/04/10 20:06:46 WARN TaskSetManager: Stage 4 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Table created with schema:
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)



## Step 1 — Schema Enforcement: Rejecting Bad Writes


In [3]:
print("=== Schema Enforcement Demo ===")
print()

# Test 1: Missing column
df_missing = df.drop("revenue")
try:
    df_missing.write.format("delta").mode("append").save(TABLE)
except Exception as e:
    print(f"❌ Missing column rejected:")
    print(f"   {str(e)[:120]}...")
print()

# Test 2: Wrong type
df_wrong_type = df.withColumn("order_id", F.col("order_id").cast("string"))
try:
    df_wrong_type.write.format("delta").mode("append").save(TABLE)
except Exception as e:
    print(f"❌ Wrong type rejected:")
    print(f"   {str(e)[:120]}...")
print()

# Test 3: Extra column
df_extra = df.withColumn("new_col", F.lit("extra"))
try:
    df_extra.write.format("delta").mode("append").save(TABLE)
except Exception as e:
    print(f"❌ Extra column rejected:")
    print(f"   {str(e)[:120]}...")
print()

# Test 4: Correct schema — accepted
df.limit(100).write.format("delta").mode("append").save(TABLE)
print(f"✅ Correct schema accepted. Rows: {spark.read.format('delta').load(TABLE).count():,}")

=== Schema Enforcement Demo ===



26/04/10 20:06:52 WARN TaskSetManager: Stage 5 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
                                                                                


❌ Wrong type rejected:
   [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'order_id' and 'order_id'...

❌ Extra column rejected:
   [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: e8563120-71c4-4997...



26/04/10 20:06:53 WARN TaskSetManager: Stage 6 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:06:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 10:==================================================>     (45 + 4) / 50]

✅ Correct schema accepted. Rows: 200,100


## Step 2 — mergeSchema: Opt-In Evolution

When you intentionally ADD columns to the source data,
use `mergeSchema=True` to evolve the table schema.


In [4]:
# Add new columns to source data
df_v2 = df.withColumn("discount_pct",   F.round(F.rand(42) * 0.3, 2)) \
           .withColumn("loyalty_points", (F.col("revenue") * 10).cast("int")) \
           .withColumn("channel",        F.lit("web"))

# Without mergeSchema — rejected
try:
    df_v2.limit(100).write.format("delta").mode("append").save(TABLE)
except Exception as e:
    print(f"Without mergeSchema: ❌ {type(e).__name__}")

# With mergeSchema — accepted, schema evolved
df_v2.limit(100).write.format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .save(TABLE)
print(f"With mergeSchema   : ✅ accepted")
print()
print("New schema (3 columns added):")
spark.read.format("delta").load(TABLE).printSchema()
print()
print("Old rows have null for new columns:")
spark.read.format("delta").load(TABLE) \
     .select("order_id","revenue","discount_pct","loyalty_points","channel") \
     .show(5)

Without mergeSchema: ❌ AnalysisException


26/04/10 20:06:58 WARN TaskSetManager: Stage 14 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


With mergeSchema   : ✅ accepted

New schema (3 columns added):
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- loyalty_points: integer (nullable = true)
 |-- channel: string (nullable = true)


Old rows have null for new columns:


+--------+-------+------------+--------------+-------+
|order_id|revenue|discount_pct|loyalty_points|channel|
+--------+-------+------------+--------------+-------+
|   24577|   NULL|        NULL|          NULL|   NULL|
|   24578|   NULL|        NULL|          NULL|   NULL|
|   24579|   NULL|        NULL|          NULL|   NULL|
|   24580|   NULL|        NULL|          NULL|   NULL|
|   24581|   NULL|        NULL|          NULL|   NULL|
+--------+-------+------------+--------------+-------+
only showing top 5 rows


## Step 3 — ALTER TABLE: Production Schema Changes


In [5]:
# ALTER TABLE ADD COLUMNS (explicit, no data rewrite)
spark.sql(f"""
    ALTER TABLE delta.`{TABLE}`
    ADD COLUMNS (
        return_reason  STRING  COMMENT 'Reason for return if applicable',
        rating         DOUBLE  COMMENT 'Customer rating 1-5'
    )
""")
print("Added 2 columns via ALTER TABLE:")
spark.read.format("delta").load(TABLE).printSchema()

# ALTER TABLE CHANGE COLUMN (rename with column mapping)
spark.sql(f"""
    ALTER TABLE delta.`{TABLE}`
    SET TBLPROPERTIES (
        'delta.columnMapping.mode'       = 'name',
        'delta.minReaderVersion'         = '2',
        'delta.minWriterVersion'         = '5'
    )
""")
spark.sql(f"ALTER TABLE delta.`{TABLE}` RENAME COLUMN channel TO sales_channel")
print("\nRenamed column: channel → sales_channel")
spark.read.format("delta").load(TABLE).select(
    "order_id","discount_pct","sales_channel","rating").show(3)

Added 2 columns via ALTER TABLE:
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- loyalty_points: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- return_reason: string (nullable = true)
 |-- rating: double (nullable = true)


Renamed column: channel → sales_channel


+--------+------------+-------------+------+
|order_id|discount_pct|sales_channel|rating|
+--------+------------+-------------+------+
|   24577|        NULL|         NULL|  NULL|
|   24578|        NULL|         NULL|  NULL|
|   24579|        NULL|         NULL|  NULL|
+--------+------------+-------------+------+
only showing top 3 rows


## Step 4 — Type Changes


In [6]:
# Type widening in Delta Lake requires enableTypeWidening table property
# ALTER TABLE CHANGE COLUMN for type changes needs this flag
TYPE_TABLE = f"{DATA_DIR}/07_types"
shutil.rmtree(TYPE_TABLE, ignore_errors=True)

small_schema = StructType([
    StructField("id",    IntegerType()),
    StructField("value", FloatType()),
    StructField("name",  StringType()),
])
small_df = spark.createDataFrame([(i, float(i)*1.5, f"item_{i}") for i in range(1000)], small_schema)

# Create table with type widening enabled
small_df.write.format("delta") \
        .mode("overwrite") \
        .option("delta.enableTypeWidening", "true") \
        .save(TYPE_TABLE)
print("Original types: id=INT, value=FLOAT")
spark.read.format("delta").load(TYPE_TABLE).printSchema()

# Enable type widening on existing table (if not set at write time)
spark.sql(f"ALTER TABLE delta.`{TYPE_TABLE}` SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')")

# Widen: INT → LONG (safe widening)
spark.sql(f"ALTER TABLE delta.`{TYPE_TABLE}` CHANGE COLUMN id TYPE BIGINT")
print("\nAfter ALTER COLUMN id TYPE BIGINT (safe widening INT→LONG):")

# Verify — old data readable with new type
result = spark.read.format("delta").load(TYPE_TABLE)
print(f"All {result.count()} rows readable after type change:")
result.printSchema()

print()
print("Safe type widenings (with enableTypeWidening=true):")
print("  INT → LONG    ✅")
print("  FLOAT → DOUBLE ✅")
print("  INT → DOUBLE  ✅")
print()
print("Unsafe changes (always rejected):")
print("  LONG → INT    ❌  (data loss)")
print("  DOUBLE → INT  ❌  (data loss)")
print("  STRING → INT  ❌  (incompatible)")

Original types: id=INT, value=FLOAT
root
 |-- id: integer (nullable = true)
 |-- value: float (nullable = true)
 |-- name: string (nullable = true)


After ALTER COLUMN id TYPE BIGINT (safe widening INT→LONG):
All 1000 rows readable after type change:
root
 |-- id: long (nullable = true)
 |-- value: float (nullable = true)
 |-- name: string (nullable = true)


Safe type widenings (with enableTypeWidening=true):
  INT → LONG    ✅
  FLOAT → DOUBLE ✅
  INT → DOUBLE  ✅

Unsafe changes (always rejected):
  LONG → INT    ❌  (data loss)
  DOUBLE → INT  ❌  (data loss)
  STRING → INT  ❌  (incompatible)


## Summary: Schema Management Reference

### Enforcement vs Evolution
```python
# Default: strict enforcement (recommended for production)
df.write.format("delta").mode("append").save(path)
# → Rejects: missing columns, wrong types, extra columns

# Opt-in evolution (add new columns)
df.write.format("delta").mode("append")
        .option("mergeSchema", "true").save(path)
# → Accepts: new columns (old rows get null)
```

### DDL commands
```sql
-- Add columns
ALTER TABLE delta.`/path` ADD COLUMNS (col STRING, col2 INT)

-- Rename column (requires columnMapping.mode=name)
ALTER TABLE delta.`/path`
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', ...)
ALTER TABLE delta.`/path` RENAME COLUMN old_name TO new_name

-- Safe type widening
ALTER TABLE delta.`/path` CHANGE COLUMN id id LONG
```

### Safe type changes
| From | To | Safe? |
|---|---|---|
| INT | LONG | ✅ |
| FLOAT | DOUBLE | ✅ |
| LONG | INT | ❌ overflow |
| STRING | INT | ❌ parse errors |
